# AgentCore Evaluation - 사용 예시

이 노트북에서는 AgentCore Observability의 원하는 세션 ID를 평가하는 방법을 살펴봅니다.

## 설정

In [ ]:
import os
from utils import EvaluationClient

# AWS 자격 증명 - 여기에 자격 증명을 추가합니다.
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
# os.environ['AWS_ACCESS_KEY_ID'] = ''
# os.environ['AWS_SECRET_ACCESS_KEY'] = ''
# os.environ['AWS_SESSION_TOKEN'] = ''

## 구성

In [ ]:
# AWS 구성
REGION = "us-east-1"
AGENT_ID = "strands_claude_eval-YOUR_UNIQUE_ID"
SESSION_ID = "<YOUR_SESSION_ID_HERE>"  # 평가할 세션 ID를 직접 전달합니다.

metadata = {
    "experiment": "evaluation_test",
    "description": "Testing all evaluator scopes",
}

## 클라이언트 초기화

In [ ]:
# 평가 클라이언트를 초기화합니다.
eval_client = EvaluationClient(
    region=REGION,
)

## 평가 실행

모든 evaluator 유형으로 세션을 평가하고 대시보드를 자동으로 생성합니다.

In [ ]:
# Evaluator 그룹
FLEXIBLE_EVALUATORS = [
    "Builtin.Correctness",
    "Builtin.Faithfulness",
    "Builtin.Helpfulness",
    "Builtin.ResponseRelevance",
    "Builtin.Conciseness",
    "Builtin.Coherence",
    "Builtin.InstructionFollowing",
    "Builtin.Refusal",
    "Builtin.Harmfulness",
    "Builtin.Stereotyping",
]

SESSION_ONLY_EVALUATORS = ["Builtin.GoalSuccessRate"]

SPAN_ONLY_EVALUATORS = [
    "Builtin.ToolSelectionAccuracy",
    "Builtin.ToolParameterAccuracy",
]

In [ ]:
test_groups = [
    {
        "name": "Flexible Evaluators (session scope)",
        "evaluators": FLEXIBLE_EVALUATORS,
        "scope": "session",
    },
    {
        "name": "Session-Only Evaluators",
        "evaluators": SESSION_ONLY_EVALUATORS,
        "scope": "session",
    },
    {
        "name": "Span-Only Evaluators",
        "evaluators": SPAN_ONLY_EVALUATORS,
        "scope": "span",
    },
]

all_results = []

for group in test_groups:
    try:
        results = eval_client.evaluate_session(
            session_id=SESSION_ID,
            evaluator_ids=group["evaluators"],
            agent_id=AGENT_ID,
            region=REGION,
            scope=group["scope"],
            auto_save_output=True,
            auto_create_dashboard=True,
            metadata=metadata,
        )

        print(f"\nCompleted: {len(results.results)} evaluations")
        for r in results.results:
            print(f"  {r.evaluator_name}: {r.value} - {r.label}")
            all_results.append(r)

    except Exception as e:
        print(f"\nError: {e}")

# 마무리